# Libraries

In [6]:
import sys
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
  
def save(fig, name):
    fig.savefig(OUT / "charts" / f"{name}.png")
    plt.close(fig)
 
 
def table(df, name):
    df.to_csv(OUT / "tables" / f"{name}.csv")
 
 
def header(t):
    print(f"\n{'=' * 78}\n{t}\n{'=' * 78}")

# 1. Load & Data Quality

In [10]:
from pathlib import Path
import pandas as pd

# Set your data folder
DATA = Path(".")

# Load datasets
tx = pd.read_csv(
    DATA / "building_materials_transactions_Cleaned.csv",
    parse_dates=["date"]
)

macro = pd.read_csv(
    DATA / "macro_drivers_weekly_cleaned.csv",
    parse_dates=["week"]
)

# --------------------------------------------------
# 1. DATA OVERVIEW & QUALITY
# --------------------------------------------------

print("1. DATA OVERVIEW & QUALITY")

print(
    f"Transactions : {len(tx):,} rows | "
    f"{tx['date'].min():%Y-%m-%d} -> {tx['date'].max():%Y-%m-%d} | "
    f"{tx['date'].nunique()} weekly dates | "
    f"{tx['sku'].nunique():,} SKUs"
)

print(
    f"Total revenue: ${tx['revenue'].sum():,.0f} | "
    f"units: {tx['units'].sum():,} | "
    f"avg order line: ${tx['revenue'].mean():,.0f}"
)

print(
    f"Missing values: {int(tx.isna().sum().sum())} | "
    f"duplicate rows: {int(tx.duplicated().sum())}"
)

# Check revenue calculation
mismatch = (
    tx["units"] * tx["unit_price"] - tx["revenue"]
).abs() > 0.02

print(
    f"Rows where units x unit_price != revenue: "
    f"{int(mismatch.sum())}"
)

# Check days of week
print(
    f"All transactions fall on weekday(s): "
    f"{sorted(tx['date'].dt.day_name().unique())} "
    f"-> data is weekly-bucketed, so no day-of-week analysis is possible"
)

# --------------------------------------------------
# Create time dimensions
# --------------------------------------------------

tx["quarter"] = tx["date"].dt.to_period("Q").astype(str)

tx["ym"] = (
    tx["date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

print("\nData loaded successfully.")
print(f"Transactions shape: {tx.shape}")
print(f"Macro shape: {macro.shape}")

1. DATA OVERVIEW & QUALITY
Transactions : 340,000 rows | 2019-01-07 -> 2024-12-30 | 313 weekly dates | 84,641 SKUs
Total revenue: $590,657,665 | units: 47,939,567 | avg order line: $1,737
Missing values: 0 | duplicate rows: 0
Rows where units x unit_price != revenue: 0
All transactions fall on weekday(s): ['Monday'] -> data is weekly-bucketed, so no day-of-week analysis is possible

Data loaded successfully.
Transactions shape: (340000, 17)
Macro shape: (313, 5)


# 2. Product & Category Performence

In [12]:
# ============================================================
# BUILDING MATERIALS BUSINESS ANALYSIS
# CATEGORY + SKU ANALYSIS
# ============================================================

# -----------------------------
# 1. IMPORT LIBRARIES
# -----------------------------

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter


# -----------------------------
# 2. SETUP DIRECTORIES
# -----------------------------

# Use the notebook's current working directory
BASE = Path.cwd()

# Create a proper project output folder
OUT = BASE / "business_analysis_output"

# Create subfolders
TABLES = OUT / "tables"
FIGURES = OUT / "figures"

TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("Base directory:")
print(BASE)

print("\nOutput directory:")
print(OUT)


# -----------------------------
# 3. HELPER FUNCTIONS
# -----------------------------

def header(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def table(df, name):
    """
    Save a DataFrame as a CSV inside the tables folder.
    """
    path = TABLES / f"{name}.csv"
    df.to_csv(path)
    print(f"Saved table: {path}")


def save(fig, name):
    """
    Save a matplotlib figure inside the figures folder.
    """
    path = FIGURES / f"{name}.png"
    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight"
    )
    print(f"Saved figure: {path}")


# Currency formatter
M = FuncFormatter(lambda x, pos: f"${x:,.0f}")


# -----------------------------
# 4. LOAD DATA
# -----------------------------

header("LOADING DATA")

# Transaction dataset
tx_path = BASE / "building_materials_transactions_Cleaned.csv"

# Macro dataset
macro_path = BASE / "macro_drivers_weekly_cleaned.csv"


# Check whether files exist
if not tx_path.exists():
    raise FileNotFoundError(
        f"\nTransaction file not found:\n{tx_path}\n\n"
        f"Files currently available in {BASE}:\n"
        + "\n".join(p.name for p in BASE.iterdir())
    )

if not macro_path.exists():
    raise FileNotFoundError(
        f"\nMacro file not found:\n{macro_path}\n\n"
        f"Files currently available in {BASE}:\n"
        + "\n".join(p.name for p in BASE.iterdir())
    )


# Read CSV files
tx = pd.read_csv(
    tx_path,
    parse_dates=["date"]
)

macro = pd.read_csv(
    macro_path,
    parse_dates=["week"]
)


print(f"Transactions loaded: {len(tx):,} rows")
print(f"Macro data loaded: {len(macro):,} rows")


# -----------------------------
# 5. BASIC DATA CLEANING
# -----------------------------

header("DATA CLEANING & VALIDATION")

# Make sure dates are datetime
tx["date"] = pd.to_datetime(
    tx["date"],
    errors="coerce"
)

macro["week"] = pd.to_datetime(
    macro["week"],
    errors="coerce"
)


# Numeric columns
numeric_columns = [
    "units",
    "unit_price",
    "revenue"
]

for col in numeric_columns:
    if col in tx.columns:
        tx[col] = pd.to_numeric(
            tx[col],
            errors="coerce"
        )


# Remove completely duplicated rows
duplicate_count = tx.duplicated().sum()

print(f"Duplicate rows found: {duplicate_count:,}")

if duplicate_count > 0:
    tx = tx.drop_duplicates().copy()
    print("Duplicate rows removed.")


# -----------------------------
# 6. DATA OVERVIEW
# -----------------------------

header("1. DATA OVERVIEW & QUALITY")

print(
    f"Transactions : {len(tx):,} rows | "
    f"{tx['date'].min():%Y-%m-%d} -> "
    f"{tx['date'].max():%Y-%m-%d} | "
    f"{tx['date'].nunique()} weekly dates | "
    f"{tx['sku'].nunique():,} SKUs"
)

print(
    f"Total revenue: ${tx['revenue'].sum():,.0f} | "
    f"Units: {tx['units'].sum():,} | "
    f"Average order line: ${tx['revenue'].mean():,.0f}"
)


# Missing values
missing_values = int(
    tx.isna().sum().sum()
)

print(
    f"Missing values: {missing_values:,}"
)


# Duplicate rows after cleaning
print(
    f"Duplicate rows after cleaning: "
    f"{int(tx.duplicated().sum()):,}"
)


# Revenue validation
mismatch = (
    tx["units"] * tx["unit_price"] -
    tx["revenue"]
).abs() > 0.02

print(
    f"Rows where units × unit_price != revenue: "
    f"{int(mismatch.sum()):,}"
)


# Day of week
weekday_values = sorted(
    tx["date"]
    .dropna()
    .dt.day_name()
    .unique()
)

print(
    f"Transaction weekday(s): {weekday_values}"
)

print(
    "Note: Data is weekly-bucketed, so day-of-week "
    "analysis is not meaningful."
)


# -----------------------------
# 7. CREATE TIME VARIABLES
# -----------------------------

tx["quarter"] = (
    tx["date"]
    .dt.to_period("Q")
    .astype(str)
)

tx["ym"] = (
    tx["date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)


# -----------------------------
# 8. CATEGORY ANALYSIS
# -----------------------------

header("2. TOP PRODUCTS & CATEGORIES")


cat = (
    tx.groupby("product_category")
      .agg(
          revenue=("revenue", "sum"),
          units=("units", "sum"),
          lines=("revenue", "size"),
          avg_price=("unit_price", "mean"),
          avg_line_value=("revenue", "mean"),
          skus=("sku", "nunique")
      )
      .assign(
          rev_share=lambda d:
              d["revenue"] /
              d["revenue"].sum() * 100,

          line_share=lambda d:
              d["lines"] /
              d["lines"].sum() * 100
      )
      .sort_values(
          "revenue",
          ascending=False
      )
)


# Cumulative revenue share
cat["cum_share"] = (
    cat["rev_share"].cumsum()
)


# Save category table
table(
    cat,
    "category_summary"
)


# Print category analysis
print("\nCATEGORY SUMMARY\n")

print(
    cat.round(2).to_string()
)


# -----------------------------
# 9. CATEGORY REVENUE CHART
# -----------------------------

fig, ax = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)


# Revenue by category
c = cat.sort_values(
    "revenue"
)

ax[0].barh(
    c.index,
    c["revenue"],
    color=sns.color_palette(
        "crest",
        len(c)
    )
)

ax[0].xaxis.set_major_formatter(M)

ax[0].set_title(
    "Revenue by Category"
)

ax[0].set_xlabel(
    "Revenue ($)"
)

ax[0].set_ylabel(
    "Product Category"
)


# Volume vs ticket size
ax[1].scatter(
    cat["lines"],
    cat["avg_line_value"],
    s=(
        cat["revenue"] /
        cat["revenue"].max() *
        1500
    ),
    alpha=0.6
)


# Add category labels
for k, r in cat.iterrows():

    ax[1].annotate(
        k,
        (
            r["lines"],
            r["avg_line_value"]
        ),
        fontsize=8,
        ha="center"
    )


ax[1].set_xlabel(
    "Transaction Lines (Volume)"
)

ax[1].set_ylabel(
    "Average $ per Line"
)

ax[1].set_title(
    "Volume vs. Ticket Size\n"
    "(Bubble Size = Revenue)"
)


plt.tight_layout()

save(
    fig,
    "01_category_revenue"
)

plt.show()


# -----------------------------
# 10. SKU LEVEL ANALYSIS
# -----------------------------

header("3. SKU PERFORMANCE & PARETO ANALYSIS")


sku = (
    tx.groupby(
        [
            "sku",
            "product_category"
        ]
    )
    .agg(
        revenue=("revenue", "sum"),
        units=("units", "sum"),
        lines=("revenue", "size")
    )
    .reset_index()
    .sort_values(
        "revenue",
        ascending=False
    )
)


# Cumulative revenue percentage
sku["cum_share"] = (
    sku["revenue"].cumsum() /
    sku["revenue"].sum() *
    100
)


# Rank percentage
sku["rank_pct"] = (
    np.arange(
        1,
        len(sku) + 1
    ) /
    len(sku) *
    100
)


# -----------------------------
# 11. SAVE TOP 100 SKUS
# -----------------------------

top100 = (
    sku
    .head(100)
    .set_index("sku")
)

table(
    top100,
    "top100_skus"
)


# -----------------------------
# 12. SKU CONCENTRATION
# -----------------------------

# Number of SKUs needed to reach 80% revenue
n80 = (
    sku["cum_share"] <= 80
).sum()


print(
    f"\nSKU concentration:"
)

print(
    f"Top {n80:,} SKUs "
    f"({n80 / len(sku) * 100:.1f}% of SKUs) "
    f"represent approximately 80% of revenue."
)


# Top 1% of SKUs
top_1_percent_count = max(
    1,
    int(
        len(sku) * 0.01
    )
)

top_1_percent_revenue = (
    sku["revenue"]
    .head(top_1_percent_count)
    .sum()
)

top_1_percent_share = (
    top_1_percent_revenue /
    sku["revenue"].sum() *
    100
)

print(
    f"Top 1% of SKUs = "
    f"{top_1_percent_share:.1f}% of revenue"
)


# SKUs sold only once
single_sale_skus = (
    sku["lines"] == 1
).sum()

single_sale_share = (
    sku["lines"] == 1
).mean() * 100

print(
    f"SKUs sold only once: "
    f"{single_sale_skus:,} "
    f"({single_sale_share:.1f}%)"
)


# -----------------------------
# 13. TOP 10 SKUS
# -----------------------------

print(
    "\nTOP 10 SKUs BY REVENUE:\n"
)

print(
    sku
    .head(10)
    .round(0)
    .to_string(index=False)
)


# -----------------------------
# 14. SKU PARETO CHART
# -----------------------------

fig, ax = plt.subplots(
    figsize=(8, 5)
)


ax.plot(
    sku["rank_pct"],
    sku["cum_share"],
    lw=2
)


# 80% reference line
ax.axhline(
    80,
    ls="--",
    c="grey"
)


# SKU threshold
ax.axvline(
    n80 / len(sku) * 100,
    ls="--",
    c="grey"
)


ax.set_xlabel(
    "% of SKUs (Ranked by Revenue)"
)

ax.set_ylabel(
    "Cumulative % Revenue"
)

ax.set_title(
    "SKU Pareto Curve"
)


plt.tight_layout()

save(
    fig,
    "02_sku_pareto"
)

plt.show()


# -----------------------------
# 15. TOP 3 SKUS PER CATEGORY
# -----------------------------

header("4. TOP 3 SKUs PER CATEGORY")


top_sku_by_cat = (
    sku
    .groupby(
        "product_category",
        group_keys=False
    )
    .head(3)
    .sort_values(
        [
            "product_category",
            "revenue"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# Save
table(
    top_sku_by_cat.set_index("sku"),
    "top3_sku_per_category"
)


print(
    top_sku_by_cat
    .round(2)
    .to_string(index=False)
)


# -----------------------------
# 16. CATEGORY REVENUE SHARE
# -----------------------------

header("5. CATEGORY REVENUE CONCENTRATION")


category_concentration = (
    cat[
        [
            "revenue",
            "rev_share",
            "cum_share"
        ]
    ]
    .copy()
)


print(
    category_concentration
    .round(2)
    .to_string()
)


# -----------------------------
# 17. SAVE CLEANED ANALYSIS DATA
# -----------------------------

tx_output = OUT / "transactions_analysis_ready.csv"

tx.to_csv(
    tx_output,
    index=False
)

print(
    f"\nSaved analysis-ready transaction data:"
)

print(
    tx_output
)


# -----------------------------
# 18. FINAL OUTPUT SUMMARY
# -----------------------------

header("ANALYSIS COMPLETE")

print(
    f"""
Output folder:
{OUT}

Tables created:
- category_summary.csv
- top100_skus.csv
- top3_sku_per_category.csv

Figures created:
- 01_category_revenue.png
- 02_sku_pareto.png

Additional file:
- transactions_analysis_ready.csv

Total transaction rows: {len(tx):,}
Total SKUs: {tx['sku'].nunique():,}
Total revenue: ${tx['revenue'].sum():,.0f}
Total units: {tx['units'].sum():,}
Number of categories: {tx['product_category'].nunique():,}
"""
)

Base directory:
/Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887

Output directory:
/Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output

LOADING DATA
Transactions loaded: 340,000 rows
Macro data loaded: 313 rows

DATA CLEANING & VALIDATION
Duplicate rows found: 0

1. DATA OVERVIEW & QUALITY
Transactions : 340,000 rows | 2019-01-07 -> 2024-12-30 | 313 weekly dates | 84,641 SKUs
Total revenue: $590,657,665 | Units: 47,939,567 | Average order line: $1,737
Missing values: 0
Duplicate rows after cleaning: 0
Rows where units × unit_price != revenue: 0
Transaction weekday(s): ['Monday']
Note: Data is weekly-bucketed, so day-of-week analysis is not meaningful.

2. TOP PRODUCTS & CATEGORIES
Saved table: /Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output/tables/category_summary.csv

CATEGORY SUMMARY

                         revenue     units  lines  avg_price  avg_lin

/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_11000/744920935.py:394: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/4d/j2zh3f0x41zgdh_wqkx7qvnm0000gn/T/ipykernel_11000/744920935.py:590: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Saved analysis-ready transaction data:
/Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output/transactions_analysis_ready.csv

ANALYSIS COMPLETE

Output folder:
/Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output

Tables created:
- category_summary.csv
- top100_skus.csv
- top3_sku_per_category.csv

Figures created:
- 01_category_revenue.png
- 02_sku_pareto.png

Additional file:
- transactions_analysis_ready.csv

Total transaction rows: 340,000
Total SKUs: 84,641
Total revenue: $590,657,665
Total units: 47,939,567
Number of categories: 10



# 3. Sales & Transaction Trends

In [13]:
header("3. SALES & TRANSACTION TRENDS")
yr = tx.groupby("year").agg(revenue=("revenue", "sum"), units=("units", "sum"),
                            lines=("revenue", "size"), avg_line=("revenue", "mean"))
yr["rev_yoy_%"] = yr.revenue.pct_change() * 100
yr["lines_yoy_%"] = yr.lines.pct_change() * 100
yr["avg_line_yoy_%"] = yr.avg_line.pct_change() * 100
table(yr, "yearly_summary")
print(yr.round(1).to_string())
 
monthly = tx.groupby("ym").agg(revenue=("revenue", "sum"), lines=("revenue", "size"))
fig, ax = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
ax[0].plot(monthly.index, monthly.revenue, lw=1.5)
ax[0].plot(monthly.revenue.rolling(12, center=True).mean(), lw=3, c="crimson", label="12-mo rolling avg")
ax[0].yaxis.set_major_formatter(M); ax[0].set_title("Monthly revenue"); ax[0].legend()
ax[1].plot(monthly.index, monthly.lines, c="teal"); ax[1].set_title("Monthly transaction lines")
save(fig, "03_monthly_trend")
 
# Seasonality
season = tx.groupby(["year", "month"]).revenue.sum().unstack("year")
season_idx = season.div(season.mean()).mean(axis=1)  # avg month vs its own year mean
print("\nSeasonality index (1.00 = average month):")
print(season_idx.round(2).to_string())
print(f"Peak month: {season_idx.idxmax()} ({season_idx.max():.2f}) | "
      f"Trough: {season_idx.idxmin()} ({season_idx.min():.2f})")
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(season.div(season.mean()).T, annot=True, fmt=".2f", cmap="RdYlGn", center=1, ax=ax,
            cbar_kws={"label": "vs. year average"})
ax.set_title("Monthly revenue index by year"); ax.set_xlabel("Month")
save(fig, "04_seasonality_heatmap")
 
# Category seasonality (for inventory planning)
cat_season = (tx.groupby(["product_category", "month"]).revenue.sum().unstack()
                .pipe(lambda d: d.div(d.mean(axis=1), axis=0)))
table(cat_season, "category_seasonality_index")
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(cat_season, annot=True, fmt=".2f", cmap="RdYlGn", center=1, ax=ax)
ax.set_title("Category seasonality index by month (1.0 = category's average month)")
save(fig, "05_category_seasonality")
print("\nCategory peak months:", cat_season.idxmax(axis=1).to_dict())
print("Category seasonal amplitude (max/min):",
      (cat_season.max(axis=1) / cat_season.min(axis=1)).round(2).to_dict())
 
# Category growth by year
cat_yr = tx.pivot_table(index="year", columns="product_category", values="revenue", aggfunc="sum")
cagr = ((cat_yr.loc[2024] / cat_yr.loc[2019]) ** (1 / 5) - 1) * 100
print("\nCategory revenue CAGR 2019->2024 (%):")
print(cagr.sort_values(ascending=False).round(1).to_string())
fig, ax = plt.subplots(figsize=(12, 5.5))
cat_yr.div(cat_yr.loc[2019]).mul(100).plot(ax=ax, marker="o")
ax.set_title("Category revenue indexed to 2019 = 100"); ax.legend(bbox_to_anchor=(1, 1))
save(fig, "06_category_growth")


3. SALES & TRANSACTION TRENDS
Saved table: /Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output/tables/yearly_summary.csv
          revenue    units  lines  avg_line  rev_yoy_%  lines_yoy_%  avg_line_yoy_%
year                                                                               
2019   78345461.4  7139696  53959    1451.9        NaN          NaN             NaN
2020  100853532.4  8975343  58882    1712.8       28.7          9.1            18.0
2021  104824868.9  7037964  53062    1975.5        3.9         -9.9            15.3
2022   94730445.4  7073120  54146    1749.5       -9.6          2.0           -11.4
2023  112520233.7  9468792  62270    1807.0       18.8         15.0             3.3
2024   99383123.2  8244652  57681    1723.0      -11.7         -7.4            -4.6
Saved figure: /Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output/figures/03_monthly_trend.png

Seasonalit

# 4. Customer, Channel & Regional Behavour

In [14]:
header("4. CUSTOMER, CHANNEL & REGION")
 
 
def seg(col):
    d = (tx.groupby(col).agg(revenue=("revenue", "sum"), lines=("revenue", "size"),
                             avg_line=("revenue", "mean"), avg_units=("units", "mean"))
           .assign(rev_share=lambda x: x.revenue / x.revenue.sum() * 100)
           .sort_values("revenue", ascending=False))
    table(d, f"segment_{col}")
    return d
 
 
for col in ["customer_type", "channel", "region"]:
    print(f"\n-- by {col}"); print(seg(col).round(1).to_string())
 
# customer type x category mix
mix_cust = tx.pivot_table(index="customer_type", columns="product_category", values="revenue", aggfunc="sum")
mix_cust_pct = mix_cust.div(mix_cust.sum(axis=1), axis=0) * 100
table(mix_cust_pct, "customer_category_mix_pct")
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(mix_cust_pct, annot=True, fmt=".0f", cmap="Blues", ax=ax)
ax.set_title("Category mix (% of revenue) by customer type")
save(fig, "07_customer_category_mix")
 
# region x category: index vs national mix (>1 = over-indexed)
reg_cat = tx.pivot_table(index="region", columns="product_category", values="revenue", aggfunc="sum")
reg_index = reg_cat.div(reg_cat.sum(axis=1), axis=0).div(reg_cat.sum() / reg_cat.values.sum())
table(reg_index, "region_category_index")
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(reg_index, annot=True, fmt=".2f", cmap="RdBu_r", center=1, ax=ax)
ax.set_title("Regional category over/under-index vs national mix (1.0 = national average)")
save(fig, "08_region_category_index")
flat = reg_index.stack().sort_values()
print("\nMost over-indexed region/category:", flat.tail(3).round(2).to_dict())
print("Most under-indexed region/category:", flat.head(3).round(2).to_dict())
 
# regional growth
reg_yr = tx.pivot_table(index="year", columns="region", values="revenue", aggfunc="sum")
fig, ax = plt.subplots(figsize=(12, 5.5))
reg_yr.div(reg_yr.loc[2019]).mul(100).plot(ax=ax, marker="o")
ax.set_title("Regional revenue indexed to 2019 = 100"); ax.legend(bbox_to_anchor=(1, 1))
save(fig, "09_region_growth")
 
# customer x channel
cc = tx.pivot_table(index="customer_type", columns="channel", values="revenue", aggfunc="sum")
cc_pct = cc.div(cc.sum(axis=1), axis=0) * 100
print("\nChannel split (% revenue) per customer type:"); print(cc_pct.round(1).to_string())


4. CUSTOMER, CHANNEL & REGION

-- by customer_type
Saved table: /Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output/tables/segment_customer_type.csv
                   revenue   lines  avg_line  avg_units  rev_share
customer_type                                                     
homebuilder    320007080.8  115307    2775.3      225.8       54.2
contractor     200046709.0  129067    1549.9      125.3       33.9
remodeler       52706873.1   54752     962.6       78.7        8.9
homeowner_diy   17897002.2   40874     437.9       34.9        3.0

-- by channel
Saved table: /Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output/tables/segment_channel.csv
                 revenue   lines  avg_line  avg_units  rev_share
channel                                                         
pro_dealer   367797186.8  211289    1740.7      141.1       62.3
big_box_diy  222860478.2  128711    1731.5    

# 5. Macro Drivers

In [15]:
header("5. MACRO DRIVERS vs. SALES")
wk = (tx.groupby("date").agg(revenue=("revenue", "sum"), lines=("revenue", "size"), units=("units", "sum"))
        .reset_index().merge(macro, left_on="date", right_on="week", how="left", suffixes=("", "_m")))
mcols = ["housing_starts_index", "lumber_price_index", "mortgage_rate", "season_factor"]
print("Weekly revenue correlation with macro drivers (Pearson / Spearman):")
for m in mcols:
    print(f"  {m:22s} {wk.revenue.corr(wk[m]):+.2f} / {wk.revenue.corr(wk[m], method='spearman'):+.2f}")
print("Weekly line-count correlation:")
for m in mcols:
    print(f"  {m:22s} {wk.lines.corr(wk[m]):+.2f}")
 
# lagged correlation (do macro moves lead sales?)
print("\nLagged correlation of revenue with macro driver (driver leads by N weeks):")
lag_rows = {}
for m in ["housing_starts_index", "mortgage_rate", "lumber_price_index"]:
    lag_rows[m] = [wk.revenue.corr(wk[m].shift(l)) for l in range(0, 13, 2)]
lag_df = pd.DataFrame(lag_rows, index=range(0, 13, 2)).round(2)
print(lag_df.to_string())
 
# deseasonalised view: revenue / season_factor
wk["rev_deseason"] = wk.revenue / wk.season_factor
print("\nCorrelation of DE-SEASONALISED revenue with drivers:")
for m in ["housing_starts_index", "lumber_price_index", "mortgage_rate"]:
    print(f"  {m:22s} {wk.rev_deseason.corr(wk[m]):+.2f}")
 
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
for a, m in zip(ax.ravel(), mcols):
    a.scatter(wk[m], wk.revenue, s=12, alpha=.6)
    z = np.polyfit(wk[m], wk.revenue, 1); xs = np.linspace(wk[m].min(), wk[m].max(), 50)
    a.plot(xs, np.polyval(z, xs), c="crimson")
    a.set_xlabel(m); a.set_ylabel("Weekly revenue"); a.yaxis.set_major_formatter(M)
    a.set_title(f"r = {wk.revenue.corr(wk[m]):+.2f}")
fig.suptitle("Weekly revenue vs. macro drivers", fontweight="bold", y=1.01)
save(fig, "10_macro_scatter")
 
# macro-driver time series
fig, ax = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
for a, m, col in zip(ax, ["housing_starts_index", "lumber_price_index", "mortgage_rate"],
                     ["tab:blue", "tab:green", "tab:red"]):
    a.plot(macro.week, macro[m], c=col); a.set_ylabel(m)
ax[0].set_title("Macro drivers over time")
save(fig, "11_macro_timeseries")
 
# lumber: does unit price track the lumber index?
lum = tx[tx.product_category == "lumber"].groupby("date").unit_price.mean().rename("lumber_unit_price")
lum = pd.concat([lum, macro.set_index("week").lumber_price_index], axis=1).dropna()
print(f"\nLumber avg unit price vs lumber price index: r = {lum.lumber_unit_price.corr(lum.lumber_price_index):+.2f}")
# price pass-through for all categories (price index relative to 2019 avg vs lumber index)
pp = tx.groupby(["date", "product_category"]).unit_price.mean().unstack()
pp_corr = pp.corrwith(macro.set_index("week").lumber_price_index).sort_values(ascending=False)
print("Correlation of weekly avg unit price with lumber price index, by category:")
print(pp_corr.round(2).to_string())
 
# OLS: weekly revenue ~ macro drivers (standardised)
X = wk[["housing_starts_index", "lumber_price_index", "mortgage_rate", "season_factor"]]
Xz = (X - X.mean()) / X.std(); y = wk.revenue
A = np.column_stack([np.ones(len(Xz)), Xz.values])
beta, *_ = np.linalg.lstsq(A, y, rcond=None)
pred = A @ beta
r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(f"\nOLS (standardised): R^2 = {r2:.2f}")
for n, b in zip(["intercept"] + mcols, beta):
    print(f"  {n:22s} {b:>14,.0f}")


5. MACRO DRIVERS vs. SALES
Weekly revenue correlation with macro drivers (Pearson / Spearman):
  housing_starts_index   +0.04 / +0.02
  lumber_price_index     +0.15 / +0.10
  mortgage_rate          +0.07 / +0.08
  season_factor          +0.93 / +0.94
Weekly line-count correlation:
  housing_starts_index   +0.11
  lumber_price_index     -0.11
  mortgage_rate          +0.09
  season_factor          +0.96

Lagged correlation of revenue with macro driver (driver leads by N weeks):
    housing_starts_index  mortgage_rate  lumber_price_index
0                   0.04           0.07                0.15
2                  -0.02           0.06                0.23
4                  -0.07           0.06                0.30
6                  -0.10           0.05                0.34
8                  -0.13           0.04                0.36
10                 -0.13           0.04                0.35
12                 -0.12           0.03                0.33

Correlation of DE-SEASONALISED reven

# 6. Anomaly Detection

In [16]:
header("6. ANOMALIES & OUTLIERS")
 
# 6a. Weekly revenue anomalies: robust z-score on deviation from seasonal-trend expectation
wk = wk.sort_values("date").reset_index(drop=True)
wk["trend"] = wk.revenue.rolling(52, center=True, min_periods=26).median()
wk["woy"] = wk.date.dt.isocalendar().week.astype(int).clip(upper=52)
wk["resid"] = wk.revenue / wk.trend
wk["seasonal"] = wk.groupby("woy").resid.transform("median")
wk["adj"] = wk.resid / wk.seasonal
med = wk.adj.median(); mad = (wk.adj - med).abs().median() * 1.4826
wk["rz"] = (wk.adj - med) / mad
anom_w = wk[wk.rz.abs() > 3][["date", "revenue", "lines", "rz"]].sort_values("rz")
table(anom_w.set_index("date"), "weekly_revenue_anomalies")
print(f"Weekly revenue anomalies (|robust z| > 3): {len(anom_w)} of {len(wk)} weeks")
print(anom_w.round(2).to_string(index=False) if len(anom_w) else "  none - weekly sales track trend x season tightly")
print("Most extreme weeks (robust z):")
print(wk.reindex(wk.rz.abs().sort_values(ascending=False).index).head(5)[["date","revenue","rz"]].to_string(index=False, float_format="%.2f"))
 
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(wk.date, wk.revenue, lw=1, label="Weekly revenue")
ax.plot(wk.date, wk.trend, c="orange", lw=2, label="52-wk rolling median")
ax.scatter(anom_w.date, anom_w.revenue, c="red", zorder=5, label="Anomaly")
ax.yaxis.set_major_formatter(M); ax.legend(); ax.set_title("Weekly revenue with anomalies flagged")
save(fig, "12_weekly_anomalies")
 
# 6b. Structural breaks: biggest YoY swings by category x year
swing = cat_yr.pct_change() * 100
print("\nLargest category YoY swings (%):")
print(swing.stack().sort_values(key=abs, ascending=False).head(8).round(1).to_string())
 
# 6c. Transaction-level outliers (within category, IQR on log revenue & units)
def iqr_flag(s, k=3.0):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return (s < q1 - k * i) | (s > q3 + k * i)
 
 
tx["log_rev"] = np.log(tx.revenue); tx["log_units"] = np.log(tx.units)
tx["out_rev"] = tx.groupby("product_category").log_rev.transform(lambda s: iqr_flag(s))
tx["out_units"] = tx.groupby("product_category").log_units.transform(lambda s: iqr_flag(s))
# price outliers: unit price vs the same category in the same week
wk_cat_med = tx.groupby(["date", "product_category"]).unit_price.transform("median")
tx["price_ratio"] = tx.unit_price / wk_cat_med
tx["out_price"] = (tx.price_ratio > 1.5) | (tx.price_ratio < 0.67)
print(f"\nTransaction outliers (log-IQR x3 within category): revenue {tx.out_rev.sum():,}, units {tx.out_units.sum():,}")
print(f"Price outliers (>1.5x or <0.67x category-week median): {tx.out_price.sum():,} "
      f"({tx.out_price.mean()*100:.2f}%)")
cols = ["date", "region", "product_category", "sku", "channel", "customer_type", "units", "unit_price", "revenue"]
flagged = tx[tx.out_rev | tx.out_units]
low_end = flagged[flagged.revenue < flagged.groupby("product_category").revenue.transform("median")]
print(f"Flagged lines are all LOW-end (tiny orders): {len(low_end)} of {len(flagged)}")
big = tx.nlargest(10, "revenue")[cols]
table(big.set_index("date"), "top10_largest_transactions")
print("\nTop 10 largest transactions (not statistical outliers within their category):"); print(big.to_string(index=False))
table(flagged[cols].set_index("date"), "flagged_outlier_transactions")
print("\nOutlier lines by customer type (share of that segment's lines):")
print((tx.groupby("customer_type")[["out_rev", "out_units", "out_price"]].mean() * 100).round(2).to_string())
print("\nOutlier price lines by category (%):")
print((tx.groupby("product_category").out_price.mean() * 100).round(2).to_string())
 
fig, ax = plt.subplots(figsize=(12, 5.5))
sns.boxplot(data=tx, x="product_category", y="log_rev", ax=ax, fliersize=1.5)
ax.set_title("Log revenue per transaction line by category"); ax.tick_params(axis="x", rotation=40)
save(fig, "13_outlier_boxplot")
 
# 6d. SKU-level price dispersion
sku_price = tx.groupby("sku").unit_price.agg(["mean", "std", "count"])
sku_price = sku_price[sku_price["count"] >= 5]
sku_price["cv"] = sku_price["std"] / sku_price["mean"]
print(f"\nSKUs with >=5 sales: {len(sku_price):,} | median price CV: {sku_price.cv.median():.2f} | "
      f"p95 CV: {sku_price.cv.quantile(.95):.2f}")


6. ANOMALIES & OUTLIERS
Saved table: /Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output/tables/weekly_revenue_anomalies.csv
Weekly revenue anomalies (|robust z| > 3): 0 of 313 weeks
  none - weekly sales track trend x season tightly
Most extreme weeks (robust z):
      date    revenue    rz
2021-05-24 4313188.00  2.92
2019-03-25 1017230.28 -2.46
2021-04-26 3369872.45  2.44
2021-05-03 3819993.53  2.43
2020-09-14 2920169.47  2.31
Saved figure: /Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output/figures/12_weekly_anomalies.png

Largest category YoY swings (%):
year  product_category  
2023  paint_coatings        36.7
      electrical            35.6
      plumbing              34.7
2020  lumber                33.3
      drywall               32.4
      plumbing              30.6
      windows_doors         30.1
2023  hardware_fasteners    28.7

Transaction outliers (log-IQR x3 within cate

# Key Takeways

In [17]:
header("8. AUTO-GENERATED HEADLINES")
print(f"- Top category: {cat.index[0]} ({cat.rev_share.iloc[0]:.1f}% of revenue); "
      f"top 3 = {cat.rev_share.head(3).sum():.1f}%.")
print(f"- Fastest growing category 2019-24: {cagr.idxmax()} ({cagr.max():+.1f}% CAGR); "
      f"slowest: {cagr.idxmin()} ({cagr.min():+.1f}%).")
print(f"- Peak season month: {season_idx.idxmax()}, trough: {season_idx.idxmin()} "
      f"(swing {season_idx.max()/season_idx.min():.2f}x).")
print(f"- Best revenue year: {yr.revenue.idxmax()}; weakest: {yr.revenue.idxmin()}.")
print(f"- Strongest macro correlate of weekly revenue: "
      f"{max(mcols, key=lambda m: abs(wk.revenue.corr(wk[m])))}.")
print(f"- {len(anom_w)} anomalous weeks and {int((tx.out_rev | tx.out_units).sum()):,} outlier transaction lines flagged.")
print(f"\nAll charts -> {OUT/'charts'} | tables -> {OUT/'tables'}")


8. AUTO-GENERATED HEADLINES
- Top category: lumber (37.0% of revenue); top 3 = 66.0%.
- Fastest growing category 2019-24: lumber (+5.3% CAGR); slowest: hardware_fasteners (+3.8%).
- Peak season month: 7, trough: 1 (swing 5.27x).
- Best revenue year: 2023; weakest: 2019.
- Strongest macro correlate of weekly revenue: season_factor.
- 0 anomalous weeks and 34 outlier transaction lines flagged.

All charts -> /Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output/charts | tables -> /Users/jahaankachwala/anaconda_projects/418a81ac-ba87-4faf-aad5-317bda212887/business_analysis_output/tables
